# 🗺️ Topic models: finding themes nobody labelled
### BERTopic on the advice-forum posts

**Module 2 · Session 01 · notebook 3 of 3**

In notebook 2 every post came with a label. Usually it doesn't: customer reviews, support tickets, survey
answers, news articles arrive as a pile of text. The question becomes *what is this pile about?*

A **topic model** groups documents by theme and describes each group with words. **BERTopic** does it in four
steps, each one something you already know:

1. **Embed** every post (sentence embeddings, notebook 2).
2. **Reduce** the 384 dimensions to a few with UMAP (dimensionality reduction, module 1).
3. **Cluster** the reduced points with HDBSCAN (clustering, module 1). Each cluster is a topic.
4. **Describe** each cluster with its most characteristic words (a TF-IDF variant, notebook 1).

We keep the labels hidden from the model, and only use them at the end to check what it found.

Run all cells (Runtime → Run all). Installing BERTopic takes a minute or two. If Colab asks for access to a secret called `HF_TOKEN`, click *Cancel*: the models are public.

Contents
1. Data and embeddings
2. Building the topic model
3. What did it find?
4. Maps of the topics
5. Topics vs the labels we hid
6. Naming topics with an LLM, then checking
7. Takeaways

In [ ]:
# @title Setup: libraries { display-mode: "form" }
%pip install -q bertopic datamapplot
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bertopic, sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
print("bertopic", bertopic.__version__, "· sentence-transformers", sentence_transformers.__version__)

---
# 📬 1 · Data and embeddings

The same 457 posts as in notebook 2. We embed them with the same model. With 457 short posts this takes less
than a minute on a CPU.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/aaubs/ds-master/main/data/dk_posts_synth_en_processed.json"
df = pd.read_json(DATA_URL)
df["text"] = df["title"] + " . " + df["selftext"]
docs = df["text"].tolist()

encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(docs, show_progress_bar=True)
print(len(docs), "posts ·", embeddings.shape)

---
# ⚙️ 2 · Building the topic model

Each of the four steps is a component we choose and configure:

* **UMAP** `n_neighbors=10, n_components=5`: squeeze 384 dimensions into 5, keeping local neighbourhoods.
  `random_state` makes the result repeatable (UMAP is random otherwise).
* **HDBSCAN** `min_cluster_size=10`: a topic needs at least 10 posts. Posts that fit no dense cluster are put
  in topic **−1**, the outliers. That is a feature: not every post has to belong somewhere.
  `cluster_selection_method="leaf"` asks for many small, tight clusters. With the default (`"eom"`), our
  test runs often put about 400 of the 457 posts into one giant topic. Try it and see.
* **CountVectorizer**: which words may describe a topic. We drop English stopwords plus forum filler words, and
  keep words that appear in at least two posts.
* **MaximalMarginalRelevance**: picks describing words that fit the topic but are not near-duplicates of each
  other, so you get *bus, phone, cinema, sidewalk* rather than *friend, friends, friendship*.

In [ ]:
forum_filler = ["im", "ive", "dont", "don", "just", "like", "really", "know", "feel", "think", "want", "get", "going",
                "time", "people", "thing", "things", "hey", "hi", "thanks", "advice", "help", "especially",
                "quite", "say", "talk", "new", "does", "doesn", "make", "way", "tips"]
vectorizer = CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS) + forum_filler, ngram_range=(1, 2), min_df=2)
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="leaf",
                        prediction_data=True)

topic_model = BERTopic(embedding_model=encoder, umap_model=umap_model, hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer, representation_model=MaximalMarginalRelevance(diversity=0.3), top_n_words=10)
topics, _ = topic_model.fit_transform(docs, embeddings)
df["topic"] = topics
print(f"{df['topic'].nunique() - (-1 in topics)} topics · {(df['topic'] == -1).sum()} outlier posts")

---
# 🔎 3 · What did it find?

One row per topic: its size, the describing words, and a few typical posts.

In [ ]:
info = topic_model.get_topic_info()
info[["Topic", "Count", "Representation"]]

In [ ]:
# Read the typical posts of one topic: change the number
topic_id = 0
for d in topic_model.get_representative_docs(topic_id):
    print("•", d[:220], "\n")

With these settings you should see 10 to 12 topics, and around a quarter of the posts in topic −1. The exact
numbers shift a little between computers: UMAP is not perfectly repeatable across machines.

🤔 *Pick two topics. Read their typical posts. Would you give them the same name the keywords suggest?*

---
# 🗺️ 4 · Maps of the topics

Three views. Hover over the interactive ones.

In [ ]:
topic_model.visualize_barchart(top_n_topics=12, n_words=6)

In [ ]:
# Every post as a point; nearby points have similar meaning
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric="cosine", random_state=42).fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=umap_2d, hide_annotations=True)

In [ ]:
# A static topic map, good for a report or a slide
fig = topic_model.visualize_document_datamap(docs, reduced_embeddings=umap_2d)

---
# 🧪 5 · Topics vs the labels we hid

The posts do have labels; the model never saw them. A crosstab shows how the topics it discovered line up with
the categories a human chose.

In [ ]:
names = dict(zip(info["Topic"], info["Name"]))
ct = pd.crosstab(df["topic"].map(names), df["label"])
ct.style.background_gradient(cmap="Purples", axis=None)

Read it row by row:
* A row with one dark cell: the topic **is** one of the labels, found without being told.
* A row spread over several labels: the topic cuts across the categories, maybe a theme the labels missed
  (money worries show up in housing *and* in relationships).
* A label spread over several rows: the model split a broad category into finer themes.

Neither view is "right". Labels answer the question someone asked in advance; topics show what is in the text.

---
# 🏷️ 6 · Naming topics with an LLM, then checking

Keyword lists like `2_stand_bus_phone_cinema` are hard to present. An LLM can turn keywords and a few posts
into a readable name. In session 2 we do this from code; here you do it by hand, which makes it easier to see
what can go wrong.

### ✏️ Try it
The cell builds a prompt with the keywords and three typical posts for each topic. Paste it into a chat
assistant.

In [ ]:
parts = []
for t in info["Topic"]:
    if t == -1:
        continue
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
    posts = "\n".join("- " + d[:200] for d in topic_model.get_representative_docs(t)[:3])
    parts.append(f"Topic {t}\nKeywords: {words}\nTypical posts:\n{posts}")
prompt = ("Give each topic below a short, specific name (3-6 words). Return one line per topic: "
          "'Topic N: name'.\n\n" + "\n\n".join(parts))
print(prompt)

**Then check the names:**
* Compare each name with the crosstab in section 5: does a topic named after *dating* really sit mostly in
  *Love & Dating*?
* Is any name more specific than the posts justify (the LLM saw only three posts per topic)?
* Do two topics get almost the same name? Then maybe they should be merged
  (`topic_model.merge_topics`), or the model found a distinction the name misses.

---
# 🧰 7 · Takeaways

* BERTopic is a pipeline of familiar parts: embeddings → UMAP → HDBSCAN → TF-IDF-style keywords. Each part has
  knobs, and the result depends on them. Try `min_cluster_size=5` or `20` in section 2 and see how the topics
  change.
* A topic model does not find "the true topics". It finds *one useful grouping*, and you decide whether it is
  useful by reading posts, not by trusting the keywords.
* Outliers (topic −1) are honest: forcing every document into a topic makes topics worse. If you do need every
  post assigned, `topic_model.reduce_outliers(docs, topics)` moves each outlier to its closest topic.
* LLMs make topics readable, and add a new thing to check.

**In session 2** we let LLMs do more of the work: extract structured information, classify without training
data, and name topics from code. Everything you saw today is the yardstick for judging that.